# 🚀 Vibe Coding LLM Fine-Tuning (Qwen2.5-Coder-7B + QLoRA)

Fine-tunes `Qwen/Qwen2.5-Coder-7B-Instruct` on Google Colab's **Free T4 GPU** using standard **HuggingFace Transformers + PEFT + TRL + BitsAndBytes QLoRA 4-Bit**.

### 📌 Before Running:
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run All**
3. Paste your Hugging Face token when prompted in the last cell

In [ ]:
# ============================================
# CELL 1: Verify GPU is Connected
# ============================================
!nvidia-smi
import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    raise RuntimeError("❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# ============================================
# CELL 2: Install Required Packages
# All packages have pre-built wheels. NO compilation needed.
# ============================================
!pip install -q bitsandbytes accelerate peft trl datasets huggingface_hub
print("\n✅ All packages installed successfully!")

In [ ]:
# ============================================
# CELL 3: Load Qwen2.5-Coder-7B in 4-Bit Quantization
# ============================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"

# 4-bit quantization config (reduces VRAM from 16GB → ~5.5GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {model_name} in 4-bit precision...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer.pad_token = tokenizer.eos_token
model.config.use_cache = False
print(f"✅ {model_name} loaded in 4-bit! VRAM used: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# ============================================
# CELL 4: Inject QLoRA Adapters into Model Layers
# ============================================
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

# LoRA adapter configuration
lora_config = LoraConfig(
    r=16,                    # Rank dimension (higher = more capacity, more VRAM)
    lora_alpha=32,           # Scaling factor
    lora_dropout=0.05,       # Dropout for regularization
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
trainable_params, total_params = model.get_nb_trainable_parameters()
print(f"✅ QLoRA adapters injected!")
print(f"   Trainable parameters: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

In [ ]:
# ============================================
# CELL 5: Download & Format Vibe Coding Dataset
# ============================================
import json
from datasets import Dataset

!wget -q -O vibe_coding_dataset.json https://raw.githubusercontent.com/shawaz03/LLM/main/data/vibe_coding_dataset.json

with open("vibe_coding_dataset.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Format into ChatML token structure
formatted_samples = []
for item in raw_data:
    text = (
        f"<|im_start|>system\n{item['system']}<|im_end|>\n"
        f"<|im_start|>user\n{item['instruction']}<|im_end|>\n"
        f"<|im_start|>assistant\n{item['response']}<|im_end|>"
    )
    formatted_samples.append({"text": text})

dataset = Dataset.from_list(formatted_samples)
print(f"✅ Loaded & formatted {len(dataset)} ChatML samples!")
print(f"   Sample preview (first 200 chars): {formatted_samples[0]['text'][:200]}...")

In [ ]:
# ============================================
# CELL 6: Fine-Tune with SFTTrainer
# Training takes ~25-35 minutes on T4 GPU
# ============================================
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    optim="adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    save_strategy="no",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print("🚀 Starting QLoRA Fine-Tuning (60 steps, ~25-35 min on T4)...")
print("   Watch the loss value decrease below — that means the model is learning!\n")
trainer.train()
print("\n🎉 Training Complete!")

In [ ]:
# ============================================
# CELL 7: Save LoRA Adapter Weights Locally
# ============================================
output_dir = "vibe_coder_lora"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"✅ LoRA adapter saved to '{output_dir}/' folder!")

import os
for f in os.listdir(output_dir):
    size = os.path.getsize(os.path.join(output_dir, f))
    print(f"   📄 {f} ({size / 1024:.1f} KB)")

In [ ]:
# ============================================
# CELL 8: Upload to Hugging Face Model Hub
# ============================================
from huggingface_hub import notebook_login, HfApi

print("🔑 Paste your Hugging Face WRITE token below:")
print("   (Get one from: https://huggingface.co/settings/tokens)\n")
notebook_login()

repo_id = "shawaz03/vibe-coder-7b"
api = HfApi()
api.create_repo(repo_id, exist_ok=True)
api.upload_folder(folder_path="vibe_coder_lora", repo_id=repo_id)
print(f"\n🎉 SUCCESS! Your fine-tuned Vibe Coder model is LIVE at:")
print(f"   👉 https://huggingface.co/{repo_id}")